# Iloczyn skalarny źle uwarunkowanych wektorów: quire

Etap 4 planu. x = [u, u, d], y = [v, −v, 1] w losowej kolejności: dokładny wynik to suma(d),
a składniki u·v rosną ze skalą, więc uwarunkowanie Σ|xᵢyᵢ| / |x·y| rośnie. Funkcje: `examples/quire_dot.py`.
Na KV260 posit32 (i quire) liczy sprzęt: `USE_HW = True` (overlay `posit32_vec` z etapu 4).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
sys.path.insert(0, str(pathlib.Path.cwd().parent / "examples"))
import numpy as np, matplotlib.pyplot as plt
import positnp as pn
import quire_dot

In [ ]:
USE_HW = False
if USE_HW:
    from positnp.backend_hw import PositVecHw
    pn.set_backend(PositVecHw.open())

In [ ]:
rows = quire_dot.experiment(scales=[10.0 ** k for k in range(0, 17)])
cond = [c for _, c, _ in rows]
fig, ax = plt.subplots(figsize=(9, 4.5))
for name in rows[0][2]:
    err = [max(r[name], 1e-18) for _, _, r in rows]
    ax.loglog(cond, err, marker="o", label=name)
ax.axhline(2.0 ** -28, color="gray", ls=":", label="½ ulp posit32 przy 1")
ax.set_xlabel("uwarunkowanie Σ|xᵢyᵢ| / |x·y|"); ax.set_ylabel("błąd względny"); ax.legend(); ax.grid(alpha=.3, which="both")

Quire dodaje każdy iloczyn dokładnie i zaokrągla raz, więc błąd posit32 z quire zostaje na poziomie
pół ulp niezależnie od uwarunkowania (dopóki składniki mieszczą się w quire, |xᵢyᵢ| ≤ maxpos²).
float64 traci wszystkie cyfry przy uwarunkowaniu ~10¹⁶, float32 i posit32 bez quire dużo wcześniej.

In [ ]:
x = pn.array([1e8, 1.0, -1e8])
float(np.sum(x)), float(np.sum(np.array([1e8, 1.0, -1e8], np.float32))), pn.quire(x)